# 04 — Tessera annual embeddings

Samples the Tessera Zarr store with `GeoTesseraZarr.sample_points()` — point-only, no tile download (fetching all 511 tiles would be ~300 GB). Year coverage is measured per tile, not assumed from `gz.years`, using `MIN_POINT_COVERAGE` and `MAX_COVERAGE_SPREAD` thresholds.

Output: `03_Features/Tessera/tessera_{years}_features.parquet`. Runtime: <10 min.

## Setup

In [1]:
!pip -q install -U geotessera geopandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.9/258.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 38.5 MB/s eta 0:00:00


In [2]:
import os, json, glob, time
import numpy as np
import pandas as pd
import geopandas as gpd

from google.colab import drive
drive.mount('/content/drive')

import geotessera
from geotessera.store import GeoTesseraZarr
print("geotessera:", geotessera.__version__)

ROOT = '/content/drive/MyDrive/Crop_Classification'
DIR_POINTS   = f'{ROOT}/01_Points'
DIR_FEATURES = f'{ROOT}/02_RawTimeSeries'
DIR_QA       = f'{ROOT}/05_QA'
PART_DIR     = f'{DIR_FEATURES}/Tessera/_partials'
os.makedirs(PART_DIR, exist_ok=True)

POINTS_GPKG  = f'{DIR_POINTS}/wbcrop_points_extended.gpkg'
POINTS_LAYER = 'wbcrop_points_extended'

Mounted at /content/drive
geotessera: 0.10.2


## Configuration

In [3]:
CANDIDATE_YEARS = [2022, 2023, 2024, 2025]   # tested below; only those with data are used
BATCH = 5000                                  # points per sample_points call
MAX_RETRIES = 3

gz = GeoTesseraZarr()
print("Years in the store schema:", gz.years)

Years in the store schema: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [4]:
pts = gpd.read_file(POINTS_GPKG, layer=POINTS_LAYER).to_crs('EPSG:4326')
assert pts['id'].is_unique, "duplicate ids in the points file"

# Sorting by location keeps each batch inside a small area, so the store reads fewer
# chunks per call. Same points, fewer round trips.
pts = pts.sort_values(['longitude', 'latitude']).reset_index(drop=True)
coords = list(zip(pts.geometry.x.astype(float), pts.geometry.y.astype(float)))
print(f"Points: {len(pts):,}")

Points: 74,351


## Which years hold data, tile by tile

Coverage is per-tile, not per-region — probes one point per tile per candidate year.

In [5]:
GRID_RES = 0.1
pts['tile_lon'] = np.floor(pts.geometry.x / GRID_RES) * GRID_RES + GRID_RES / 2
pts['tile_lat'] = np.floor(pts.geometry.y / GRID_RES) * GRID_RES + GRID_RES / 2

# One representative point from each tile
reps = pts.groupby(['tile_lon', 'tile_lat']).first().reset_index()
rep_coords = list(zip(reps.geometry.x.astype(float), reps.geometry.y.astype(float)))
print(f"Tiles: {len(reps)}   (one probe point each)\n")

Tiles: 511   (one probe point each)



In [6]:
tile_cov = {}
for y in CANDIDATE_YEARS:
    if y not in gz.years:
        print(f"  {y}  not in schema")
        continue
    try:
        X = gz.sample_points(rep_coords, year=y)
        ok = np.isfinite(np.asarray(X)).any(axis=1)          # tile has data
        tile_cov[y] = ok
        print(f"  {y}  tiles with data: {ok.sum():>4}/{len(ok)} ({ok.mean():>5.1%})")
    except Exception as e:
        print(f"  {y}  FAIL  {type(e).__name__}: {str(e)[:60]}")

  2022  tiles with data:   39/511 ( 7.6%)
  2023  tiles with data:   46/511 ( 9.0%)
  2024  tiles with data:  511/511 (100.0%)
  2025  tiles with data:    0/511 ( 0.0%)


In [7]:
# Translate tile coverage into POINT coverage, and check it against the classes.
tile_key = pts[['tile_lon', 'tile_lat']].apply(tuple, axis=1)
reps_key = reps[['tile_lon', 'tile_lat']].apply(tuple, axis=1)

summary = []
for y, ok in tile_cov.items():
    good_tiles = set(reps_key[ok])
    covered = tile_key.isin(good_tiles)
    by_crop = covered.groupby(pts['crop']).mean()
    summary.append({'year': y,
                    'tiles_%': round(float(ok.mean()) * 100, 1),
                    'points_%': round(float(covered.mean()) * 100, 1),
                    'worst_crop': by_crop.idxmin(),
                    'worst_%': round(float(by_crop.min()) * 100, 1),
                    'best_%': round(float(by_crop.max()) * 100, 1)})
    globals()[f'covered_{y}'] = covered

cov_tab = pd.DataFrame(summary).set_index('year')
cov_tab['spread_pp'] = (cov_tab['best_%'] - cov_tab['worst_%']).round(1)
print(cov_tab.to_string())

      tiles_%  points_%  worst_crop  worst_%  best_%  spread_pp
year                                                           
2022      7.6       9.2  pine_apple      0.0    47.5       47.5
2023      9.0      10.7  pine_apple      0.0    41.1       41.1
2024    100.0     100.0   aman_rice    100.0   100.0        0.0
2025      0.0       0.0   aman_rice      0.0     0.0        0.0


### Choosing the years

`MIN_POINT_COVERAGE` drops mostly-missing years; `MAX_COVERAGE_SPREAD` drops years whose missingness itself separates classes.

In [8]:
MIN_POINT_COVERAGE = 90.0    # %
MAX_COVERAGE_SPREAD = 10.0   # percentage points between best and worst crop

YEARS = []
for y, r in cov_tab.iterrows():
    if r['points_%'] < MIN_POINT_COVERAGE:
        print(f"  {y}  rejected: only {r['points_%']}% of points covered")
    elif r['spread_pp'] > MAX_COVERAGE_SPREAD:
        print(f"  {y}  rejected: coverage varies {r['spread_pp']} pp across crops "
              f"(worst {r['worst_crop']} at {r['worst_%']}%)")
    else:
        YEARS.append(int(y))
        print(f"  {y}  accepted: {r['points_%']}% covered, spread {r['spread_pp']} pp")

print(f"\nUsable years: {YEARS}")
assert YEARS, ("No year passes both tests. Lower MIN_POINT_COVERAGE only if you are "
               "prepared to report the gap, and never lower MAX_COVERAGE_SPREAD.")

N_DIM = None

  2022  rejected: only 9.2% of points covered
  2023  rejected: only 10.7% of points covered
  2024  accepted: 100.0% covered, spread 0.0 pp
  2025  rejected: only 0.0% of points covered

Usable years: [2024]


## Timing test

Measure one batch before committing to the full run.

In [9]:
t0 = time.time()
X = gz.sample_points(coords[:BATCH], year=YEARS[0])
dt = time.time() - t0

N_DIM = X.shape[1]
n_batches = int(np.ceil(len(coords) / BATCH)) * len(YEARS)

print(f"batch of {BATCH:,}: {dt:.1f}s   shape {X.shape}   finite {np.isfinite(X).mean():.1%}")
print(f"dimensions: {N_DIM}")
print(f"\nProjected: {n_batches} batches x {dt:.1f}s = {n_batches * dt / 60:.1f} minutes")
print(f"value range: {np.nanmin(X):.4f} to {np.nanmax(X):.4f}")

batch of 5,000: 4.8s   shape (5000, 128)   finite 100.0%
dimensions: 128

Projected: 15 batches x 4.8s = 1.2 minutes
value range: -11.0412 to 10.9375


## Extraction

One parquet per batch per year — a disconnect costs one batch.

In [10]:
def run_batch(year, start):
    out = os.path.join(PART_DIR, f"b_{year}_{start:06d}.parquet")
    if os.path.exists(out):
        return 'skipped'

    sl = slice(start, start + BATCH)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            X = gz.sample_points(coords[sl], year=year)
            break
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"  {year} batch {start} FAILED: {type(e).__name__}: {e}")
                return 'error'
            time.sleep(3 * attempt)

    block = pd.DataFrame(np.asarray(X, dtype='float32'),
                         columns=[f'TE{i:03d}_{year}' for i in range(X.shape[1])])
    block.insert(0, 'id', pts['id'].values[sl])
    block.to_parquet(out, index=False)
    return 'done'

In [11]:
t0 = time.time()
stats = {'done': 0, 'skipped': 0, 'error': 0}
jobs = [(y, s) for y in YEARS for s in range(0, len(coords), BATCH)]

for i, (year, start) in enumerate(jobs, 1):
    stats[run_batch(year, start)] += 1
    if i % 5 == 0 or i == len(jobs):
        el = time.time() - t0
        print(f"[{i}/{len(jobs)}] {stats} | {el/60:.1f} min, "
              f"~{(len(jobs)-i)*el/max(i,1)/60:.1f} min left")

[5/15] {'done': 5, 'skipped': 0, 'error': 0} | 0.4 min, ~0.8 min left
[10/15] {'done': 10, 'skipped': 0, 'error': 0} | 0.7 min, ~0.4 min left
[15/15] {'done': 15, 'skipped': 0, 'error': 0} | 1.1 min, ~0.0 min left


## Merge

In [12]:
te = None
for year in YEARS:
    parts = sorted(glob.glob(f'{PART_DIR}/b_{year}_*.parquet'))
    exp = int(np.ceil(len(coords) / BATCH))
    print(f"{year}: {len(parts)} / {exp} batches")
    assert len(parts) == exp, f"{year} is missing batches — re-run the extraction cell"

    d = pd.concat([pd.read_parquet(f) for f in parts], ignore_index=True)
    assert d['id'].is_unique, f"{year}: duplicate ids"
    te = d if te is None else te.merge(d, on='id', how='outer')

emb_cols = [c for c in te.columns if c != 'id']
print(f"\nCombined: {te.shape}   ({N_DIM} dims x {len(YEARS)} year(s))")
assert te['id'].is_unique

2024: 15 / 15 batches

Combined: (74351, 129)   (128 dims x 1 year(s))


## Quality checks

In [13]:
print(f"Points in {len(pts):,} | out {te['id'].nunique():,}")
print(f"NaN fraction : {te[emb_cols].isna().mean().mean():.3%}")

all_nan = te[emb_cols].isna().all(axis=1)
print(f"Rows entirely NaN: {int(all_nan.sum()):,} ({all_nan.mean():.2%})")

const = [c for c in emb_cols if te[c].nunique() <= 1]
print(f"Constant dimensions: {len(const)}", const[:8] if const else "")

print("\nFirst 5 dimensions:")
print(te[emb_cols[:5]].describe().round(4).T[['mean', 'std', 'min', 'max']].to_string())

Points in 74,351 | out 74,351
NaN fraction : 0.000%
Rows entirely NaN: 0 (0.00%)
Constant dimensions: 0 

First 5 dimensions:
              mean     std     min      max
TE000_2024  5.4863  1.4262 -1.1047  10.6473
TE001_2024 -0.9640  1.8798 -7.5487   7.7791
TE002_2024 -0.6799  1.6018 -5.6631   6.1748
TE003_2024  2.8082  1.3289 -2.2333   9.6217
TE004_2024  1.5391  1.3554 -4.8034   6.4149


In [14]:
# Missingness must not track the class. Tessera coverage is generated by sampling
# regions, so a gap can be spatial — and space correlates with crop.
# crop comes from the points table itself (no separate labels file)
m = te[['id']].assign(nan_frac=te[emb_cols].isna().mean(axis=1)) \
              .merge(pts[['id', 'crop']], on='id', how='left')
by_crop = m.groupby('crop')['nan_frac'].mean().sort_values(ascending=False)
print("Missing fraction by crop:")
print(by_crop.round(4).to_string())

spread = float(by_crop.max() - by_crop.min())
print(f"\nSpread: {spread:.2%}")
print("*** NON-RANDOM — coverage gaps track the class. ***" if spread > 0.05
      else "Uniform across classes.")

Missing fraction by crop:
crop
aman_rice     0.0
aus_rice      0.0
banana        0.0
betel_leaf    0.0
boro_rice     0.0
flower        0.0
groundnut     0.0
jute          0.0
maize         0.0
mustard       0.0
others        0.0
pine_apple    0.0
potato        0.0
sugarcane     0.0
tea           0.0
tobacco       0.0
vegetables    0.0
wheat         0.0

Spread: 0.00%
Uniform across classes.


## Separability, for comparison with the other sources

In [15]:
PERENNIAL = ['banana', 'betel_leaf', 'pine_apple', 'sugarcane', 'tea']
lab = pts.set_index('id')['crop']
X = te.set_index('id')[emb_cols]
per = lab.reindex(X.index).isin(PERENNIAL)

d = ((X[per.values].mean() - X[~per.values].mean()).abs() /
     np.sqrt((X[per.values].var() + X[~per.values].var()) / 2))
print("Top 10 dimensions:")
print(d.sort_values(ascending=False).head(10).round(3).to_string())
print(f"\nmean d {d.mean():.3f} | max {d.max():.3f}")
print("\nMeasured so far: S1 filtered max 1.96, S2 max 1.79, AlphaEarth max 2.32.")
print("This is a one-dimensional indicator. Random Forest combines weak features that")
print("Cohen's d scores individually, so the ablation table is what decides.")

Top 10 dimensions:
TE013_2024    1.739
TE016_2024    1.648
TE118_2024    1.500
TE029_2024    1.490
TE052_2024    1.481
TE011_2024    1.444
TE005_2024    1.370
TE067_2024    1.365
TE127_2024    1.342
TE020_2024    1.276

mean d 0.596 | max 1.739

Measured so far: S1 filtered max 1.96, S2 max 1.79, AlphaEarth max 2.32.
This is a one-dimensional indicator. Random Forest combines weak features that
Cohen's d scores individually, so the ablation table is what decides.


## Save

In [16]:
YEAR_TAG = '_'.join(str(y) for y in YEARS)
FEATURES_OUT = f'{DIR_FEATURES}/Tessera/tessera_{YEAR_TAG}_features.parquet'
DICT_CSV     = f'{DIR_FEATURES}/Tessera/tessera_data_dictionary.csv'

te.drop(columns=const, errors='ignore').to_parquet(FEATURES_OUT, index=False)
print("Saved:", FEATURES_OUT)

pd.DataFrame([{'column': c, 'source': 'tessera',
               'variable': c.rsplit('_', 1)[0],
               'variable_meaning': 'Tessera annual embedding dimension',
               'statistic': f"year {c.rsplit('_', 1)[1]}",
               'retained': c not in const,
               'cohens_d': round(float(d.get(c, np.nan)), 4)}
              for c in emb_cols]).to_csv(DICT_CSV, index=False)

with open(f'{DIR_QA}/40_tessera_qa.json', 'w') as f:
    json.dump({'generated': pd.Timestamp.now().isoformat(),
               'geotessera_version': geotessera.__version__,
               'method': 'GeoTesseraZarr.sample_points',
               'schema_years': list(gz.years), 'years_with_data': YEARS,
               'points_in': int(len(pts)), 'points_out': int(te['id'].nunique()),
               'dimensions': len(emb_cols), 'dims_per_year': int(N_DIM),
               'constant_dimensions': const,
               'nan_fraction': float(te[emb_cols].isna().mean().mean()),
               'missing_by_crop': by_crop.round(5).to_dict(),
               'missing_spread': spread,
               'max_cohens_d': float(d.max()), 'mean_cohens_d': float(d.mean())},
              f, indent=2, default=str)
print("QA saved.")

Saved: /content/drive/MyDrive/Crop_Classification/02_RawTimeSeries/Tessera/tessera_2024_features.parquet
QA saved.


---
### Next: `90_Master_Assembly.ipynb`
- `gz.years` is the schema, not coverage — always measure.
- If only one year passes, note which seasons it misses (kharif, if 2024-only).
- Compare against the matching AlphaEarth year only.
- Record the `geotessera` version.